# EDA 6 · Summary: hypotheses, findings, features

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import mannwhitneyu, wilcoxon

from src.config import FEATURES_DIR, TOPIC_NOISE, TOPIC_SHORT, game_names
from src.db import read
from src.features.anomalies import group_events, review_anomalies
from src.features.daily import collection, daily_series, daily_topics, label_counts, launch_window, online_players
from src.features.monthly import month_changes, monthly_panel
from src.text.topics import FIXED_LABELS
from src.viz.charts import use_style

use_style()
TECH = ["Crashes, bugs and performance", "Servers and connection", "Won't launch or log in", "Lost saves",
        "Launchers, accounts and anti-cheat"]  # technical complaints (labels from topics/02_explore_name)
ONLINE = {"Online PvP", "MMO"}  # Steam categories of online games (H7)
MIN_MONTH_REVIEWS = 100   # a game-month counts with at least this many reviews (stable shares)

In [2]:
names = game_names()
collected = collection()
games = collected.index[collected["complete"].ne(False)]
series = daily_series(games)
online = online_players(games)
topic_counts = daily_topics(games)
by_label = label_counts(topic_counts)
days = series.index.get_level_values("day")
pd.Series({
    "games (fully collected)": len(games),
    "kept reviews": f"{series['n'].sum():,}",
    "with a topic": f"{topic_counts.to_numpy().sum():,}",
    "review types (labels)": by_label.shape[1],
    "first review": f"{days.min():%Y-%m-%d}",
    "last full day": f"{days.max():%Y-%m-%d}",
    "game-days": f"{len(series):,}",
}).to_frame("value")

,value
games (fully collected),62
kept reviews,"25,022,355"
with a topic,"25,022,355"
review types (labels),47
first review,2010-10-15
last full day,2026-11-08
game-days,"149,676"


## Monthly panel

The hypotheses are about online players, and SteamCharts has their full history only as monthly averages, so they are checked per game and month: the change from the previous month (log ratio for players and reviews, points for shares). A game's first and last month are left out (partial, `monthly_panel`), and months with fewer than `MIN_MONTH_REVIEWS` reviews.

In [3]:
topical = by_label.drop(columns=[FIXED_LABELS[TOPIC_SHORT], FIXED_LABELS[TOPIC_NOISE]], errors="ignore")
panel = monthly_panel(series, online, topical)
panel = panel[(panel["n"] >= MIN_MONTH_REVIEWS) & (panel["players"] > 0)]
levels = pd.DataFrame({"players": np.log(panel["players"]), "reviews": np.log(panel["n"]), "negative": 1 - panel["positive"],
                       "tech": panel[[t for t in TECH if t in panel]].sum(axis=1) / panel[topical.columns].sum(axis=1)})
d = month_changes(levels).add_prefix("d_")
months = d.index.get_level_values("month")
number = pd.Series(months.year * 12 + months.month, index=d.index)
is_next = number.groupby(level="appid").shift(-1) - number == 1
for column in ["players", "negative"]:
    d[f"next_{column}"] = d.groupby(level="appid")[f"d_{column}"].shift(-1).where(is_next)
d = d.reset_index()
d["month_id"] = d["month"].astype(str)
d = d.dropna()
for column in ["d_tech", "d_negative", "d_reviews", "d_players"]:
    d[f"z_{column[2:]}"] = (d[column] - d[column].mean()) / d[column].std()
print(f"{len(d):,} game-months in {d['appid'].nunique()} games, {d['month'].min():%Y-%m} … {d['month'].max():%Y-%m}")


def fit(formula):
    return smf.ols(formula, data=d).fit(cov_type="cluster", cov_kwds={"groups": d["appid"]})


def coef(model, name):
    return f"{model.params[name]:+.3f} (p = {model.pvalues[name]:.3f})"

4,581 game-months in 58 games, 2012-08 … 2026-09


## Hypothesis 1: technical complaints predict a drop in online players better than the overall 👎 share

Next month's change in online players regressed on this month's change in the share of technical complaints (crashes, servers, can't launch, lost saves, launchers) and in the overall 👎 share, both standardized, controlling for this month's change in reviews and players and for the calendar month. Standard errors clustered by game.

In [4]:
controls = "z_reviews + z_players + C(month_id)"
both = fit(f"next_players ~ z_tech + z_negative + {controls}")
only_tech, only_negative, base = (fit(f"next_players ~ {x} + {controls}") for x in ["z_tech", "z_negative", "1"])
h1 = pd.DataFrame({
    "coefficient (both in)": [coef(both, "z_tech"), coef(both, "z_negative")],
    "R² added alone": [only_tech.rsquared - base.rsquared, only_negative.rsquared - base.rsquared],
}, index=["technical complaints share", "overall 👎 share"])
display(h1.style.format({"R² added alone": "{:.4f}"}))

tech_p, neg_p = both.pvalues["z_tech"], both.pvalues["z_negative"]
tech_b, neg_b = both.params["z_tech"], both.params["z_negative"]
if tech_p < 0.05 and tech_b < 0 and abs(tech_b) > abs(neg_b):
    verdict = "supported: more technical complaints this month go with fewer players next month, more strongly than the overall 👎 share"
elif tech_p < 0.05 and tech_b < 0:
    verdict = "partly: technical complaints predict fewer players next month, but not more strongly than the overall 👎 share"
elif neg_p < 0.05 and neg_b < 0:
    verdict = "not supported: the overall 👎 share predicts next month's players, the technical share adds nothing clear"
else:
    verdict = "not supported: neither share clearly predicts next month's change in players"
print("H1:", verdict)

,coefficient (both in),R² added alone
technical complaints share,-0.007 (p = 0.278),0.0003
overall 👎 share,-0.009 (p = 0.326),0.0005


H1: not supported: neither share clearly predicts next month's change in players


## Hypothesis 2: online players react to a change in sentiment with a delay

Does this month's change in the 👎 share predict next month's change in players (sentiment first), and does this month's change in players predict next month's 👎 share (players first)? Same controls.

In [5]:
sentiment_first = fit(f"next_players ~ z_negative + {controls}")
players_first = fit("next_negative ~ z_players + z_reviews + z_negative + C(month_id)")
h2 = pd.DataFrame({"effect": [coef(sentiment_first, "z_negative"), coef(players_first, "z_players")]},
                  index=["👎 share this month → players next month", "players this month → 👎 share next month"])
display(h2)
s_p, s_b = sentiment_first.pvalues["z_negative"], sentiment_first.params["z_negative"]
if s_p < 0.05 and s_b < 0:
    verdict = "supported: a rise in the 👎 share is followed by fewer players the next month"
else:
    verdict = "not supported at the monthly level: a change in the 👎 share does not clearly move next month's players"
print("H2:", verdict)

,effect
👎 share this month → players next month,-0.011 (p = 0.244)
players this month → 👎 share next month,-0.001 (p = 0.546)


H2: not supported at the monthly level: a change in the 👎 share does not clearly move next month's players


## Hypothesis 3: problems show up in one language before the overall rating

Per-language daily 👎 counts are not in the series, so this checks a weaker version: are 👎 events usually driven by one language? For each event: how much each language's share of the game's reviews (by the language of the text) rose against the 4 weeks before. "One language" = one language gained at least 15 points.

In [6]:
text_columns = [c for c in series.columns if c.startswith("text_")]
first_review = series.index.to_frame(index=False).groupby("appid")["day"].min()
found = review_anomalies(series, window=28, z=4.0, exclude=launch_window(series.index, first_review))
events = group_events(found)
negative_events = events[events["kind"].isin(["bombing", "negative"])]
rows = []
for e in negative_events.itertuples():
    counts = series.loc[e.appid, text_columns]
    during = counts.loc[e.start : e.last].sum()
    before = counts.loc[e.start - pd.Timedelta(days=28) : e.start - pd.Timedelta(days=1)].sum()
    if during.sum() and before.sum():
        jump = during / during.sum() - before / before.sum()
        rows.append({"appid": e.appid, "kind": e.kind, "language": jump.idxmax().removeprefix("text_"), "jump": jump.max()})
shift = pd.DataFrame(rows)
one_language = shift["jump"] >= 0.15
print(f"{len(shift)} 👎 events (daily series): {one_language.mean():.0%} driven by one language "
      f"(its share of the reviews rose by at least 15 points)")
display(shift[one_language]["language"].value_counts().to_frame("events driven by this language"))
verdict = ("partly supported: a large share of 👎 events comes from one language's players; whether that language moves "
           "*before* the overall rating needs per-language 👎 counts by day (a possible addition to daily_series)"
           if one_language.mean() >= 0.3 else
           "not supported: most 👎 events are spread over the languages")
print("H3:", verdict)

768 👎 events (daily series): 21% driven by one language (its share of the reviews rose by at least 15 points)


,events driven by this language
language,
zh,100
en,40
ru,21


H3: not supported: most 👎 events are spread over the languages


## Hypothesis 4: review bombing is short-lived

For each review bombing event (from H3): the game's 👍 share in the 4 weeks before, during the event, and 4–8 weeks after it; and online players from the month before the event to two months after it, against the same 3-month change in all game-months. "Recovered": the 👍 share 4–8 weeks later is back within 2 points of before.

In [7]:
bombs = events[events["kind"] == "bombing"]
players = monthly_panel(series, online)["players"]


def pooled(s, start, end):
    window = s.loc[start:end, ["n", "n_positive"]].sum()
    return window["n_positive"] / window["n"] if window["n"] >= 50 else np.nan


def log_players(appid, month):
    value = players.get((appid, month), np.nan)
    return np.log(value) if value > 0 else np.nan


rows = []
for e in bombs.itertuples():
    s = series.loc[e.appid]
    if e.end + pd.Timedelta(days=56) > s.index.max():
        continue
    month_before = (e.start - pd.DateOffset(months=1)).to_period("M").to_timestamp()
    month_after = (e.last + pd.DateOffset(months=2)).to_period("M").to_timestamp()
    rows.append({"appid": e.appid, "start": e.start,
                 "before": pooled(s, e.start - pd.Timedelta(days=28), e.start - pd.Timedelta(days=1)),
                 "during": pooled(s, e.start, e.last),
                 "after": pooled(s, e.end + pd.Timedelta(days=28), e.end + pd.Timedelta(days=55)),
                 "players_change": log_players(e.appid, month_after) - log_players(e.appid, month_before)})
recovery = pd.DataFrame(rows).dropna(subset=["before", "after"])
recovery["recovered"] = recovery["after"] >= recovery["before"] - 0.02
log_p = np.log(players[players > 0])
usual = (log_p.groupby(level="appid").shift(-3) - log_p).dropna()
p_players = mannwhitneyu(recovery["players_change"].dropna(), usual).pvalue
display(pd.Series({
    "bombing events (with 8 weeks after)": len(recovery),
    "👍 share before (median)": f"{recovery['before'].median():.0%}",
    "during": f"{recovery['during'].median():.0%}",
    "4–8 weeks after": f"{recovery['after'].median():.0%}",
    "recovered": f"{recovery['recovered'].mean():.0%}",
    "players, month before → 2 months after (median change)": f"{np.expm1(recovery['players_change'].median()):+.0%}",
    "players, any 3-month span (median change)": f"{np.expm1(usual.median()):+.0%}",
    "players: bombing vs any (Mann–Whitney p)": f"{p_players:.3f}",
}).to_frame("value"))
if recovery["recovered"].mean() >= 0.5 and p_players >= 0.05:
    verdict = "supported: most bombed games get their 👍 share back within 4–8 weeks, and their players do not change differently from usual"
elif recovery["recovered"].mean() >= 0.5:
    direction = "rise" if recovery["players_change"].median() > usual.median() else "fall"
    verdict = (f"partly: the 👍 share mostly recovers within 4–8 weeks, and players {direction} more than usual around "
               "bombing (bombing comes with attention: big updates, news), so it is not a sign of players leaving"
               if direction == "rise" else
               "partly: the 👍 share mostly recovers within 4–8 weeks, but players fall more than usual around bombing")
else:
    verdict = "not supported: most bombed games keep a lower 👍 share 4–8 weeks later"
print("H4:", verdict)

,value
bombing events (with 8 weeks after),140
👍 share before (median),82%
during,60%
4–8 weeks after,80%
recovered,60%
"players, month before → 2 months after (median change)",+7%
"players, any 3-month span (median change)",-1%
players: bombing vs any (Mann–Whitney p),0.002


H4: partly: the 👍 share mostly recovers, but players change differently from usual around bombing


## Hypothesis 5: review bombers are not the game's regular players

For each review bombing event: the 👎 reviews written during it vs the game's 👎 reviews in the 28 days before. Hours played at review, how many reviews the author has written on Steam, and the share written within 2 hours of play (Steam's refund window). Paired over events (Wilcoxon signed-rank).

In [8]:
rows = []
for e in bombs.itertuples():
    r = read.reviews(appids=[e.appid], since=e.start - pd.Timedelta(days=28), until=e.end, only_kept=True, flag_columns=[],
                     columns=["created_at", "voted_up", "author_playtime_at_review", "author_num_reviews"])
    negative = r[~r["voted_up"].astype(bool)]
    during = negative["created_at"].dt.tz_convert(None) >= e.start
    if during.sum() >= 20 and (~during).sum() >= 20:
        row = {"appid": e.appid, "start": e.start}
        for part, mask in [("before", ~during), ("during", during)]:
            hours = negative.loc[mask, "author_playtime_at_review"] / 60
            row |= {f"hours_{part}": hours.median(), f"author_reviews_{part}": negative.loc[mask, "author_num_reviews"].median(),
                    f"under_2h_{part}": (hours < 2).mean()}
        rows.append(row)
bombers = pd.DataFrame(rows)
table = pd.DataFrame({
    measure: {"before (median over events)": bombers[f"{measure}_before"].median(),
              "during": bombers[f"{measure}_during"].median(),
              "events where lower during": (bombers[f"{measure}_during"] < bombers[f"{measure}_before"]).mean(),
              "Wilcoxon p": wilcoxon(bombers[f"{measure}_during"], bombers[f"{measure}_before"], nan_policy="omit").pvalue}
    for measure in ["hours", "author_reviews", "under_2h"]}).T
print(f"{len(bombers)} bombing events with at least 20 👎 reviews before and during")
display(table.style.format("{:.3g}"))
fewer_hours = table.loc["hours", "events where lower during"] > 0.5 and table.loc["hours", "Wilcoxon p"] < 0.05
quicker = bombers["under_2h_during"].median() > bombers["under_2h_before"].median() and table.loc["under_2h", "Wilcoxon p"] < 0.05
if fewer_hours or quicker:
    verdict = "supported: bombing 👎 reviews come from players with fewer hours (or more often within the refund window) than the game's usual 👎 reviews"
else:
    verdict = "not supported: bombing 👎 reviews come from players with about as many hours as the game's usual critics"
print("H5:", verdict)

138 bombing events with at least 20 👎 reviews before and during


,before (median over events),during,events where lower during,Wilcoxon p
hours,147,156,0.355,nan
author_reviews,7,7,0.522,0.0728
under_2h,0.0536,0.0489,0.543,0.601


H5: not supported: bombing 👎 reviews come from players with about as many hours as the game's usual critics


## Hypothesis 6: Steam sales bring volume, not better ratings

On Steam-wide event days vs the 28 days before each event period, per game: reviews per day and the 👍 share. Paired over game × period, games with at least 100 reviews in the 28 days before.

In [9]:
platform = pd.read_parquet(FEATURES_DIR / "platform_days.parquet")
hot = pd.Series(platform.index[platform["platform"]])
periods = hot.groupby((hot.diff() > pd.Timedelta(days=2)).cumsum()).agg(["min", "max"])
by_game = {appid: series.loc[appid] for appid in games}
rows = []
for start, last in periods.itertuples(index=False):
    for appid, s in by_game.items():
        if s.index.min() > start - pd.Timedelta(days=28) or s.index.max() < last:
            continue
        before = s.loc[start - pd.Timedelta(days=28) : start - pd.Timedelta(days=1), ["n", "n_positive"]].sum()
        during = s.loc[start:last, ["n", "n_positive"]].sum()
        if before["n"] >= 100 and during["n"] >= 20:
            rows.append({"appid": appid, "start": start,
                         "volume": (during["n"] / ((last - start).days + 1)) / (before["n"] / 28),
                         "positive_change": during["n_positive"] / during["n"] - before["n_positive"] / before["n"]})
sales = pd.DataFrame(rows)
p_positive = wilcoxon(sales["positive_change"]).pvalue
display(pd.Series({
    "game × event periods": len(sales),
    "reviews per day vs before (median)": f"{sales['volume'].median():.2f}×",
    "👍 share change (mean, points)": f"{sales['positive_change'].mean() * 100:+.1f}",
    "pairs with a lower 👍 share": f"{(sales['positive_change'] < 0).mean():.0%}",
    "👍 change ≠ 0 (Wilcoxon p)": f"{p_positive:.3f}",
}).to_frame("value"))
if sales["volume"].median() > 1.2 and (p_positive >= 0.05 or sales["positive_change"].mean() <= 0):
    verdict = ("supported: Steam-wide events bring more reviews, " +
               ("with a lower 👍 share" if p_positive < 0.05 else "without a better 👍 share"))
else:
    verdict = ("not supported, the opposite: Steam-wide events bring more reviews and a higher 👍 share (sale buyers and "
               "event participants write short praise)" if sales["volume"].median() > 1.2 else
               "not supported: they bring no clear rise in reviews")
print("H6:", verdict)

,value
game × event periods,1308
reviews per day vs before (median),1.54×
"👍 share change (mean, points)",+2.2
pairs with a lower 👍 share,34%
👍 change ≠ 0 (Wilcoxon p),0.000


H6: not supported: the 👍 share rises during them


## Hypothesis 7: language gaps are bigger in online games

Per game: how far the Chinese and the Russian 👍 share are from the English one (absolute difference, kept reviews, languages with at least 300 reviews), online games (Steam categories Online PvP or MMO: servers, cheaters, monetization) vs the rest. Mann–Whitney U.

In [10]:
profile = read.language_profile(games)
positive = profile[profile["n"] >= 300].pivot(index="appid", columns="language", values="positive")
categories = read.games().set_index("appid")["categories"]
is_online = categories.reindex(positive.index).map(lambda c: bool(ONLINE & set(c if isinstance(c, list) else [])))
rows = []
for lang in ["schinese", "russian"]:
    gap = (positive[lang] - positive["english"]).abs().dropna()
    online_gap, other_gap = gap[is_online.reindex(gap.index)], gap[~is_online.reindex(gap.index)]
    rows.append({"language": lang, "online games": len(online_gap), "other games": len(other_gap),
                 "median gap, online": online_gap.median(), "median gap, other": other_gap.median(),
                 "Mann–Whitney p (online larger)": mannwhitneyu(online_gap, other_gap, alternative="greater").pvalue})
gaps = pd.DataFrame(rows).set_index("language")
display(gaps.style.format({"median gap, online": "{:.1%}", "median gap, other": "{:.1%}", "Mann–Whitney p (online larger)": "{:.3f}"}))
larger = (gaps["median gap, online"] > gaps["median gap, other"]) & (gaps["Mann–Whitney p (online larger)"] < 0.05)
if larger.all():
    verdict = "supported: Chinese and Russian players disagree with English ones more about online games"
elif larger.any():
    names_of = {"schinese": "Chinese", "russian": "Russian"}
    verdict = (f"partly: the gap to English is larger in online games for {', '.join(names_of[l] for l in larger.index[larger])} "
               "players only")
else:
    verdict = "not supported: language gaps are no larger in online games"
print("H7:", verdict)

,online games,other games,"median gap, online","median gap, other",Mann–Whitney p (online larger)
language,,,,,
schinese,19,41,13.9%,4.3%,0.007
russian,19,42,2.1%,2.5%,0.340


H7: partly: the gap is larger in online games for schinese only


## Key findings

### Hypotheses

| | Hypothesis | Result | Evidence |
|---|---|---|---|
| H1 | Technical complaints predict a drop in players better than the 👎 share | **No** | neither predicts next month's players (−0.007, p = 0.28 and −0.009, p = 0.33) |
| H2 | Players react to sentiment with a delay | **No** | 👎 share → next month's players −0.011 (p = 0.24); daily, the 👍 share neither leads nor follows players (eda/04) |
| H3 | Problems show up in one language first | **Weak** | 21% of 768 👎 events are driven by one language (100 of them Chinese); "first" is not testable without per-language daily 👎 counts |
| H4 | Review bombing is short-lived | **Mostly** | 👍 share 82% → 60% during → 80% 4–8 weeks later, 60% of events recover; players *rise* around bombing (+7% vs −1% usual, p = 0.002): bombing comes with attention, not with players leaving |
| H5 | Bombers are not the game's regular players | **No** | their 👎 reviews come from players with as many hours (156 vs 147 median) and as many reviews written as the game's usual critics |
| H6 | Steam sales bring volume, not better ratings | **No, the opposite** | 1.5× reviews per day and +2.2 points of 👍 share (p < 0.001) on Steam-wide event days |
| H7 | Language gaps are bigger in online games | **For Chinese only** | Chinese vs English: 13.9% gap in online (PvP, MMO) games vs 4.3% in others (p = 0.007); Russian: 2.1% vs 2.5% |

**For the models**: sentiment does not move player numbers (H1, H2, H4), so forecasting players should rely on their own history, review volume, patches and Steam-wide events; sentiment is its own target. Bombing is a temporary shock (H4), and Steam-wide event days move both volume and the 👍 share (H6).

### Data and review types (eda/01, 02)
- 36% of kept reviews are too short for a type and 13% have no clear one; the rest fall into 45 types. The biggest are short praise (20%) and strong recommendation (14%).
- The most negative types: "Trash game", "Updates made it worse", detailed gameplay complaints and matchmaking (24–27% 👍); then can't launch, servers, crashes, cheaters (37–50%).
- After patches the type mix barely moves (at most ±0.3 points per type): "Updates made it worse" grows after major patches, "Strong recommendation" after hotfixes and updates.

### Time series (eda/03)
- Reviews come on weekends (+10–25% Saturday and Sunday for most games); the 👍 share has no weekday pattern.
- Levels drift (only about a third of games are stationary), day-to-day changes are stationary for almost all: models should predict changes.
- The daily 👍 share has memory in live-service games (ARC Raiders, Helldivers 2, Phasmophobia: lag-1 autocorrelation 0.85–0.9) and is noise around a stable level in single-player games (Balatro, Satisfactory, Stardew Valley: ~0).
- Steam-wide events, found from the data without a calendar: 194 days in 46 periods, every late November (Autumn sale) and late June (Summer sale); 42% of review spikes fall on them.
- 1,330 anomaly events (123 review bombing). Bombing starts within 3 days after a patch 3× as often as chance (31% vs 10% of days). The causes are readable from the types and examples: Sony's PSN account requirement (Helldivers 2), politics and DLC prices (Hearts of Iron IV, Factorio), the free-to-play switch (CS:GO).

### Patches and players (eda/04)
- Major patches bring +19% reviews in their first week (median) and −1.4 points of 👍 share on average; 25% clearly hurt it, 15% clearly help. Hotfixes and regular updates barely move it.
- Per game, patches tend to help War Thunder and No Man's Sky and to hurt Space Marine 2, Risk of Rain 2 and Trackmania.
- Online players and review counts move on the same day (correlation 0.22 of daily changes) and in the same month (0.61), with no lead either way.

### Audience (eda/05)
- 👍 share (median over games): English 89%, Russian 87%, Chinese 82%. Chinese players review after fewer hours (21 vs 33) and write much shorter reviews (15 vs 53 characters).
- The biggest language gaps: Chinese reviews of Slay the Spire 2 (−58 points vs English), Helldivers 2 (−48), Overwatch 2 (−34); Russian ones of Trackmania and MECCHA CHAMELEON (−14).
- Each language has its own complaints: English writes detailed gameplay complaints and about matchmaking, Russian "don't buy" and insults, Chinese about grind (87× more than the others), "negative until fixed", translation, servers and launching.
- Where a game is relatively expensive, its players there rate it slightly worse (Spearman −0.29 Russia, −0.23 Taiwan, −0.10 China): a weak effect.
- Shared reviewers group the games into 7 communities: strategy and factory sims; indie roguelikes and platformers; live-service action and RPG; competitive shooters; co-op horror; free-to-play classics; horror.
- A competitor's release has no consistent effect on its closest games' reviews (median +1% over 97 pairs; anecdotes both ways).

## Features for the models

What `eda/02`–`05` and the hypotheses suggest, and where each feature comes from (all in `src/features/`, saved to `data/features/`):

| Feature | Source | Why |
|---|---|---|
| Reviews per day (log), lags 1, 7, 14, 28; 7- and 28-day means | `daily_series` | strong autocorrelation of counts (eda/03) |
| Weekday | calendar | weekend +10–25% reviews for most games (eda/03); not needed for the 👍 share |
| 👍 share, 7-day and 28-day | `daily_series` | daily share is noise for stable games, persistent for live-service ones (eda/03) |
| Changes instead of levels as targets | — | levels drift, changes are stationary (eda/03) |
| Shares of review types (technical, monetization, servers, ...) | `daily_topics` + `label_counts` | what players complain about (eda/02); for sentiment targets, not for players (H1) |
| Share of reviews by text language | `daily_series` (`text_*`) | a fifth of 👎 events come from one language, mostly Chinese (H3) |
| Days since the last patch, patch size | `events.patches`, `events.days_since` | patch effects by size (eda/04) |
| Anomaly flags (bombing, spike), days since bombing | `anomalies.review_anomalies`, `group_events` | outliers to mark or down-weight; bombing fades within 4–8 weeks (H4) |
| Steam-wide event days | `anomalies.platform_days` | 1.5× reviews and +2 points of 👍 share on those days (H6) |
| Online players (daily since ~Aug 2026, monthly before) | `online_players`, `monthly.monthly_panel` | target and feature; reviews move with players, sentiment does not lead them (eda/04, H2) |
| Shared-audience neighbours, their reviews and releases | `audience.audience_overlap`, `neighbours` | communities of games (eda/05); game embeddings |
| Language profile of the game (share RU / EN / ZH, their 👍 gaps) × online game | `read.language_profile`, Steam categories | audiences rate differently (eda/05); the Chinese gap is 3× larger in online games (H7) |
